In [ ]:
!pip install rasterio 
!pip install fiona
!pip install rasterstats
!pip install raster4ml
!pip install sklearn
!pip install shapely

In [1]:
import numpy as np
import fiona
import rasterio.mask
from matplotlib import pyplot
import pandas as pd
import glob
import os
import rasterio
from rasterio.crs import CRS
from rasterio.plot import show
from raster4ml.extraction import batch_extract_by_polygons
import os
import gc
import shutil


os.chdir(r'C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Cahn\CHACABUCO\CHACABUCO\LARGO1\\')

def create_directory(dir, name):
    # Directory
    directory = name

    # Parent Directory path
    parent_dir = dir

    # Path
    path = os.path.join(parent_dir, directory)

    # Create the directory
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)

In [2]:
def get_bands(pixel_values):
    bands = dict.fromkeys(['blue', 'green', 'red', 'rede', 'nir'], [])
    bands['blue'] = pixel_values[0]
    bands['green'] = pixel_values[1]
    bands['red'] = pixel_values[2]
    bands['rede'] = pixel_values[3]
    bands['nir'] = pixel_values[4]
    return bands

def generate_indexes(dir, profile, bands):
    profile.update(
        dtype=rasterio.float32,
        count=1)
  
    create_directory(dir, "INDEXES")
    with rasterio.open(dir+"/INDEXES/blue.tif", 'w', **profile) as rst:
        rst.write_band(1, bands['blue'].astype('float32'))
        del rst
        gc.collect()

    with rasterio.open(dir+"/INDEXES/green.tif", 'w', **profile) as rst:
        rst.write_band(1, bands['green'].astype('float32'))
        del rst
        gc.collect()

    with rasterio.open(dir+"/INDEXES/red.tif", 'w', **profile) as rst:
        rst.write_band(1, bands['red'].astype('float32'))
        del rst
        gc.collect()

    with rasterio.open(dir+"/INDEXES/rede.tif", 'w', **profile) as rst:
        rst.write_band(1, bands['rede'].astype('float32'))
        del rst
        gc.collect()

    with rasterio.open(dir+"/INDEXES/nir.tif", 'w', **profile) as rst:
        rst.write_band(1, bands['nir'].astype('float32'))
        del rst
        gc.collect()

    NDVI = (bands['nir'] - bands['red']) / (bands['nir'] + bands['red'])
    with rasterio.open(dir+"/INDEXES/NDVI.tif", 'w', **profile) as rst:
        rst.write_band(1, NDVI.astype('float32'))
        del rst
        gc.collect()

    GNDVI = (bands['nir']-bands['green'])/(bands['nir']+bands['green'])
    with rasterio.open(dir+"/INDEXES/GNDVI.tif", 'w', **profile) as rst:
        rst.write_band(1, GNDVI.astype('float32'))
        del rst
        gc.collect()

    RVI_1 = bands['nir']/bands['red']
    with rasterio.open(dir+"/INDEXES/RVI_1.tif", 'w', **profile) as rst:
        rst.write_band(1, RVI_1.astype('float32'))
        del rst
        gc.collect()

    GCI = (bands['nir']/bands['green'])-1.0
    with rasterio.open(dir+"/INDEXES/GCI.tif", 'w', **profile) as rst:
        rst.write_band(1, GCI.astype('float32'))
        del rst
        gc.collect()

    RGVI = bands['red']/bands['green']
    with rasterio.open(dir+"/INDEXES/RGVI.tif", 'w', **profile) as rst:
        rst.write_band(1, RGVI.astype('float32'))
        del rst
        gc.collect()

    DVI = bands['nir']-bands['red']
    with rasterio.open(dir+"/INDEXES/DVI.tif", 'w', **profile) as rst:
        rst.write_band(1, DVI.astype('float32'))
        del rst
        gc.collect()
    
    L = 0.5
    SAVI = ((bands['nir']-bands['red'])/(bands['nir']+bands['red']+L))*(1.0+L)
    with rasterio.open(dir+"/INDEXES/SAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, SAVI.astype('float32'))
        del rst
        gc.collect()

    MSAVI = 0.5*((2.0*bands['nir'])+1.0-np.sqrt(np.square(2.0*bands['nir']+1.0)-8.0*(bands['nir']-bands['red'])))
    with rasterio.open(dir+"/INDEXES/MSAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, MSAVI.astype('float32'))
        del rst
        gc.collect()

    OSAVI = (bands['nir']-bands['red'])/(bands['nir']+bands['red']+0.16)
    with rasterio.open(dir+"/INDEXES/OSAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, OSAVI.astype('float32'))
        del rst
        gc.collect()

    RDVI = np.sqrt((np.square(bands['nir']-bands['red']))/(bands['nir']+bands['red']))
    with rasterio.open(dir+"/INDEXES/RDVI.tif", 'w', **profile) as rst:
        rst.write_band(1, RDVI.astype('float32'))
        del rst
        gc.collect()

    TVI = 60.0*(bands['nir']-bands['green'])-100.0*(bands['red']-bands['green'])
    with rasterio.open(dir+"/INDEXES/TVI.tif", 'w', **profile) as rst:
        rst.write_band(1, TVI.astype('float32'))
        del rst
        gc.collect()

    a = 0.96916 
    b = 0.084726
    TSAVI = (a*(bands['nir']-a*bands['red']-b))/(a*bands['nir']+bands['red']-a*b)
    with rasterio.open(dir+"/INDEXES/TSAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, TSAVI.astype('float32'))
        del rst
        gc.collect()

    PVI = (bands['nir']-a*bands['red']-b)/np.sqrt(1+np.square(a))
    with rasterio.open(dir+"/INDEXES/TSAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, TSAVI.astype('float32'))
        del rst
        gc.collect()

    SAVI_2 = bands['nir']/(bands['red']-(b/a))
    with rasterio.open(dir+"/INDEXES/SAVI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, SAVI_2.astype('float32'))
        del rst
        gc.collect()

    X = 0.08
    ATSAVI= (a*(-a*bands['red']-b))/(a*bands['nir']+bands['red']-a*b+X*(1+np.square(a)))
    with rasterio.open(dir+"/INDEXES/ATSAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, ATSAVI.astype('float32'))
        del rst
        gc.collect()

    NDWI = (bands['green']-bands['nir'])/(bands['green']+bands['nir'])
    with rasterio.open(dir+"/INDEXES/NDWI.tif", 'w', **profile) as rst:
        rst.write_band(1, NDWI.astype('float32'))
        del rst
        gc.collect()

    NPCI = (bands['red']-bands['blue'])/(bands['red']+bands['blue'])
    with rasterio.open(dir+"/INDEXES/NPCI.tif", 'w', **profile) as rst:
        rst.write_band(1, NPCI.astype('float32'))
        del rst
        gc.collect()

    SRPI = bands['blue']/bands['red']
    with rasterio.open(dir+"/INDEXES/SRPI.tif", 'w', **profile) as rst:
        rst.write_band(1, SRPI.astype('float32'))
        del rst
        gc.collect()

    RVI_2 = bands['nir']/bands['green']
    with rasterio.open(dir+"/INDEXES/RVI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, RVI_2.astype('float32'))
        del rst
        gc.collect()

    MCARI = (bands['rede']-bands['red']-0.2*(bands['rede']-bands['green']))*(bands['rede']/bands['red'])
    with rasterio.open(dir+"/INDEXES/MCARI.tif", 'w', **profile) as rst:
        rst.write_band(1, MCARI.astype('float32'))
        del rst
        gc.collect()

    MCARI_1 = 1.2*(2.5*(bands['nir']-bands['red'])-1.3*(bands['nir']-bands['green']))
    with rasterio.open(dir+"/INDEXES/MCARI_1.tif", 'w', **profile) as rst:
        rst.write_band(1, MCARI_1.astype('float32'))
        del rst
        gc.collect()

    MCARI_2 = 1.5*(2.5*(bands['nir']-bands['red'])-1.3*(bands['nir']-bands['green']))*(np.square(2.0*bands['nir']+1))-(6.0*bands['nir']-5.0*bands['red'])-0.5
    with rasterio.open(dir+"/INDEXES/MCARI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, MCARI_2.astype('float32'))
        del rst
        gc.collect()

    MTVI_1 = 1.2*(1.2*(bands['nir']-bands['green'])-2.5*(bands['red']-bands['green']))
    with rasterio.open(dir+"/INDEXES/MTVI_1.tif", 'w', **profile) as rst:
        rst.write_band(1, MTVI_1.astype('float32'))
        del rst
        gc.collect()

    MTVI_2 = 1.5*(1.2*(bands['nir']-bands['green'])-2.5*(bands['red']-bands['green']))*(np.square(2*bands['nir']+1))-(6.0*bands['nir']-5.0*bands['red'])-0.5
    with rasterio.open(dir+"/INDEXES/MTVI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, MTVI_2.astype('float32'))
        del rst
        gc.collect()

    R_MCARI_MTVI2 = ((bands['rede']-bands['red']-0.2*(bands['rede']-bands['green']))*(bands['rede']/bands['red']))/(1.5*(1.2*(bands['nir']-bands['green'])-2.5*(bands['red']-bands['green']))*(np.square(2*bands['nir']+1))-(6.0*bands['nir']-5.0*bands['red'])-0.5)
    with rasterio.open(dir+"/INDEXES/R_MCARI_MTVI2.tif", 'w', **profile) as rst:
        rst.write_band(1, R_MCARI_MTVI2.astype('float32'))
        del rst
        gc.collect()

    EVI = (bands['nir']-bands['red'])/(bands['nir']+6.0*bands['red']-7.5*bands['blue']+1.0)
    with rasterio.open(dir+"/INDEXES/EVI.tif", 'w', **profile) as rst:
        rst.write_band(1, EVI.astype('float32'))
        del rst
        gc.collect()

    DATT = (bands['nir']-bands['rede'])/(bands['nir']-bands['red'])
    with rasterio.open(dir+"/INDEXES/DATT.tif", 'w', **profile) as rst:
        rst.write_band(1, DATT.astype('float32'))
        del rst
        gc.collect()

    NDCI = (bands['rede']-bands['green'])/(bands['rede']+bands['green'])
    with rasterio.open(dir+"/INDEXES/NDCI.tif", 'w', **profile) as rst:
        rst.write_band(1, NDCI.astype('float32'))
        del rst
        gc.collect()

    PSRI = (bands['red']-bands['green'])/bands['rede']
    with rasterio.open(dir+"/INDEXES/PSRI.tif", 'w', **profile) as rst:
        rst.write_band(1, PSRI.astype('float32'))
        del rst
        gc.collect()

    SIPI = (bands['nir']-bands['blue'])/(bands['nir']+bands['red'])
    with rasterio.open(dir+"/INDEXES/SIPI.tif", 'w', **profile) as rst:
        rst.write_band(1, SIPI.astype('float32'))
        del rst
        gc.collect()

    SPVI = 0.4*3.7*(bands['nir']-bands['red'])-1.2*np.absolute(bands['green']-bands['red'])
    with rasterio.open(dir+"/INDEXES/SPVI.tif", 'w', **profile) as rst:
        rst.write_band(1, SPVI.astype('float32'))
        del rst
        gc.collect()

    TCARI = 3.0*((bands['rede']-bands['red'])-0.2*(bands['rede']-bands['green'])*(bands['rede']/bands['red']))
    with rasterio.open(dir+"/INDEXES/TCARI.tif", 'w', **profile) as rst:
        rst.write_band(1, TCARI.astype('float32'))
        del rst
        gc.collect()

    R_TCARI_OSAVI = (3.0*((bands['rede']-bands['red'])-0.2*(bands['rede']-bands['green'])*(bands['rede']/bands['red'])))/((bands['nir']-bands['red'])/(bands['nir']+bands['red']+0.16))
    with rasterio.open(dir+"/INDEXES/R_TCARI_OSAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, R_TCARI_OSAVI.astype('float32'))
        del rst
        gc.collect()

    RERI = (bands['rede']-bands['red'])/bands['nir']
    with rasterio.open(dir+"/INDEXES/RERI.tif", 'w', **profile) as rst:
        rst.write_band(1, RERI.astype('float32'))
        del rst
        gc.collect()

    NDRE = (bands['nir']-bands['rede'])/(bands['nir']+bands['rede'])
    with rasterio.open(dir+"/INDEXES/NDRE.tif", 'w', **profile) as rst:
        rst.write_band(1, NDRE.astype('float32'))
        del rst
        gc.collect()

    MTCI = (bands['nir']-bands['rede'])/(bands['rede']-bands['red'])
    with rasterio.open(dir+"/INDEXES/MTCI.tif", 'w', **profile) as rst:
        rst.write_band(1, MTCI.astype('float32'))
        del rst
        gc.collect()

    EVI_2 = 2.5*((bands['nir']-bands['red'])/(bands['nir']+2.4*bands['red']+1.0))
    with rasterio.open(dir+"/INDEXES/EVI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, EVI_2.astype('float32'))
        del rst
        gc.collect()

    RECI = (bands['nir']/bands['rede'])-1
    with rasterio.open(dir+"/INDEXES/RECI.tif", 'w', **profile) as rst:
        rst.write_band(1, RECI.astype('float32'))
        del rst
        gc.collect()

    NEXG = (2*bands['green']-bands['red']-bands['blue'])/(bands['green']+bands['red']+bands['blue'])
    with rasterio.open(dir+"/INDEXES/NEXG.tif", 'w', **profile) as rst:
        rst.write_band(1, NEXG.astype('float32'))
        del rst
        gc.collect()

    NGRDI = (bands['green']-bands['red'])/(bands['green']+bands['red'])
    with rasterio.open(dir+"/INDEXES/NGRDI.tif", 'w', **profile) as rst:
        rst.write_band(1, NGRDI.astype('float32'))
        del rst
        gc.collect()

    ENDVI = (bands['nir']+bands['green']-2.0*bands['blue'])/(bands['nir']+bands['green']+2.0*bands['blue'])
    with rasterio.open(dir+"/INDEXES/ENDVI.tif", 'w', **profile) as rst:
        rst.write_band(1, ENDVI.astype('float32'))
        del rst
        gc.collect()

    ARI_2 = bands['nir']*((1.0/bands['green'])-(1.0/bands['rede']))
    with rasterio.open(dir+"/INDEXES/ARI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, ARI_2.astype('float32'))
        del rst
        gc.collect()

    CRI_2 = (1.0/bands['green'])-(1.0/bands['rede'])
    with rasterio.open(dir+"/INDEXES/CRI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, CRI_2.astype('float32'))
        del rst
        gc.collect()

    #HERE WE CAN VISUALIZE THE INDEX MAPS
    #pyplot.imshow(NDVI)
    #pyplot.show()
    
    values = batch_extract_by_polygons(dir+"/INDEXES", 
                                       r"C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Cahn\CHACABUCO\CHACABUCO\LARGO1\PLOTBOUNDARIES\PLOTBOUNDARIES_PROJECT.shp",
                                       "ID", statistics=['mean'])
    print(values)
    create_directory(dir, "VALUES")
    values.to_csv(dir+"/VALUES/values.csv")
    #CALCULATE INDEXES AND CREATE A RASTER OF EACH OF THEM, THEN SAVE THEM INTO A FOLDER

In [3]:

    #SAVE THE VALUES FOR THE INDEXES INTO A CSV FILE
    #values = batch_extract_by_polygons(r"E:\JUANCRUZ\THESIS\IVESDALE\INDEXES",
    #                               r'E:\JUANCRUZ\THESIS\IVESDALE\PRYT_53016_53037_yield.shp',
    #                                  "plot_id", statistics=['mean'])
    #values.to_csv(r"E:\JUANCRUZ\THESIS\IVESDALE\VALUES\values.csv")
    #CALCULATE INDEXES AND CREATE A RASTER OF EACH OF THEM, THEN SAVE THEM INTO A FOLDER

In [4]:
def normalize_pixels(pixel_values):
    pixel_values = (pixel_values - pixel_values.min()) / (pixel_values.max() - pixel_values.min()) 
    return pixel_values

"""
Considering the Chacabuco field and its 6 dates datase
1. For each date:
1.1 Open the raster images
1.2 Get the pixels
1.3 Normalize pixel values
1.4 Get de bands values
1.5 Save specifc bands .tif images
1.6 Calculate indexes
"""

dir_path = './'
ignore_dir = "INDEXES"
# Walks through the subdirectories and list files with .tif extension
for root, dirs, files in os.walk(dir_path):
    print(dirs)
    if ignore_dir in dirs:
        print('entrou nesse carai')
        dirs.remove("INDEXES")
        continue

    for file in files:
        if file.endswith(".tif"):
            file_path = os.path.join(root, file)
            # Open the file here
            with rasterio.open(file_path) as f:
                print(f"File {file_path} opened with success!")

                pixel_values = f.read()
                profile = f.profile
                del f
                gc.collect()

                pixel_values = normalize_pixels(pixel_values)
                bands = get_bands(pixel_values)
                for chave in bands.keys():
                    print(chave)
                del pixel_values
                gc.collect()

                generate_indexes(root, profile, bands)
                del profile
                del bands
                gc.collect()
                
                """
                image_norm = (pixel_values - pixel_values.min()) / (pixel_values.max() - pixel_values.min()) 
                del image_norm
                gc.collect()

                Now we can use the show function from rasterio, passing in the image to display it.
                Note that this function expects the numpy array to be either a float ranging from 0 to 1, or an uint8 ranging from 0 to 255. 
                Since our image is an uint16, we first normalize in order for it to render properly.
                show(image_norm)
                """
print("tomar no cy")

['08-30-2021', '09-24-2021', '10-05-2021', '10-07-2021', '10-26-2021', '11-17-2021', 'PLOTBOUNDARIES']
[]
File ./08-30-2021\LARGO1_0830.tif opened with success!
blue
green
red
rede
nir


<ipython-input-2-350fa6669687>:41: RuntimeWarning: invalid value encountered in true_divide
  NDVI = (bands['nir'] - bands['red']) / (bands['nir'] + bands['red'])
<ipython-input-2-350fa6669687>:47: RuntimeWarning: invalid value encountered in true_divide
  GNDVI = (bands['nir']-bands['green'])/(bands['nir']+bands['green'])
<ipython-input-2-350fa6669687>:53: RuntimeWarning: divide by zero encountered in true_divide
  RVI_1 = bands['nir']/bands['red']
<ipython-input-2-350fa6669687>:53: RuntimeWarning: invalid value encountered in true_divide
  RVI_1 = bands['nir']/bands['red']
<ipython-input-2-350fa6669687>:59: RuntimeWarning: divide by zero encountered in true_divide
  GCI = (bands['nir']/bands['green'])-1.0
<ipython-input-2-350fa6669687>:59: RuntimeWarning: invalid value encountered in true_divide
  GCI = (bands['nir']/bands['green'])-1.0
<ipython-input-2-350fa6669687>:65: RuntimeWarning: divide by zero encountered in true_divide
  RGVI = bands['red']/bands['green']
<ipython-input-2-35

  0%|                                                                                           | 0/47 [00:00<?, ?it/s]

shape1
      name  range  plot data_refer  Shape_Leng    Shape_Area  OBJECTID  \
0      1-1    1.0   1.0        1-1    0.000132  6.249197e-10        26   
1      1-2    1.0   2.0        1-2    0.000132  6.369651e-10        29   
2      1-3    1.0   3.0        1-3    0.000132  6.445363e-10        36   
3      1-4    1.0   4.0        1-4    0.000133  6.731808e-10        28   
4      1-5    1.0   5.0        1-5    0.000132  6.324303e-10        47   
..     ...    ...   ...        ...         ...           ...       ...   
295  20-11   20.0  11.0      20-11    0.000133  6.612961e-10       291   
296  20-12   20.0  12.0      20-12    0.000132  6.511900e-10       289   
297  20-13   20.0  13.0      20-13    0.000132  6.519197e-10       296   
298  20-14   20.0  14.0      20-14    0.000131  6.221089e-10       292   
299  20-15   20.0  15.0      20-15    0.000131  6.429254e-10       269   

         FIELD Name_1 CICLE HEALTH_STA  YIELD   ID  \
0    CHACABUCO   None  None    AVARAGE   4404    0

  2%|█▊                                                                                 | 1/47 [00:00<00:37,  1.24it/s]

...

shape1
      name  range  plot data_refer  Shape_Leng    Shape_Area  OBJECTID  \
0      1-1    1.0   1.0        1-1    0.000132  6.249197e-10        26   
1      1-2    1.0   2.0        1-2    0.000132  6.369651e-10        29   
2      1-3    1.0   3.0        1-3    0.000132  6.445363e-10        36   
3      1-4    1.0   4.0        1-4    0.000133  6.731808e-10        28   
4      1-5    1.0   5.0        1-5    0.000132  6.324303e-10        47   
..     ...    ...   ...        ...         ...           ...       ...   
295  20-11   20.0  11.0      20-11    0.000133  6.612961e-10       291   
296  20-12   20.0  12.0      20-12    0.000132  6.511900e-10       289   
297  20-13   20.0  13.0      20-13    0.000132  6.519197e-10       296   
298  20-14   20.0  14.0      20-14    0.000131  6.221089e-10       292   
299  20-15   20.0  15.0      20-15    0.000131  6.429254e-10       269   

         FIELD Name_1 CICLE HEALTH_STA  YIELD   ID  \
0    CHACABUCO   None  None    AVARAGE   4404    0

 98%|████████████████████████████████████████████████████████████████████████████████▎ | 46/47 [00:46<00:01,  1.00s/it]

shape1
      name  range  plot data_refer  Shape_Leng    Shape_Area  OBJECTID  \
0      1-1    1.0   1.0        1-1    0.000132  6.249197e-10        26   
1      1-2    1.0   2.0        1-2    0.000132  6.369651e-10        29   
2      1-3    1.0   3.0        1-3    0.000132  6.445363e-10        36   
3      1-4    1.0   4.0        1-4    0.000133  6.731808e-10        28   
4      1-5    1.0   5.0        1-5    0.000132  6.324303e-10        47   
..     ...    ...   ...        ...         ...           ...       ...   
295  20-11   20.0  11.0      20-11    0.000133  6.612961e-10       291   
296  20-12   20.0  12.0      20-12    0.000132  6.511900e-10       289   
297  20-13   20.0  13.0      20-13    0.000132  6.519197e-10       296   
298  20-14   20.0  14.0      20-14    0.000131  6.221089e-10       292   
299  20-15   20.0  15.0      20-15    0.000131  6.429254e-10       269   

         FIELD Name_1 CICLE HEALTH_STA  YIELD   ID  \
0    CHACABUCO   None  None    AVARAGE   4404    0

100%|██████████████████████████████████████████████████████████████████████████████████| 47/47 [00:47<00:00,  1.00s/it]


     ARI_2_mean  ATSAVI_mean  blue_mean  CRI_2_mean  DATT_mean  DVI_mean  \
0      0.591331    -0.401190   0.241516    1.531239   0.315794  0.220015   
1      0.508731    -0.422673   0.281688    1.378133        inf  0.193925   
2      0.509993    -0.432943   0.255943    1.404240   0.421955  0.171624   
3      0.445438    -0.441971   0.287537    1.215606   0.311259  0.158302   
4      0.570295    -0.416792   0.267587    1.449089        inf  0.196784   
..          ...          ...        ...         ...        ...       ...   
295    0.554418    -0.434078   0.230777    1.500897   0.442132  0.179427   
296    0.637493    -0.406886   0.224709    1.638145   0.402895  0.217900   
297    0.529834    -0.432277   0.223232    1.482801   0.446574  0.177994   
298    0.552679    -0.427847   0.212230    1.601535   0.426103  0.182255   
299    0.362530    -0.476816   0.242914    1.101489        inf  0.116118   

     ENDVI_mean      EVI_mean  EVI_2_mean  GCI_mean  ...  R_MCARI_MTVI2_mean  \
0      

In [ ]:
"""
import pandas as pd
df = pd.read_csv(r"E:\JUANCRUZ\THESIS\IVESDALE\VALUES\values.csv") 
df

df.describe()

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import geopandas as gpd
import numpy as np

## Feature & Target Selection
X = df[[col for col in df.columns if col!='yield']]
y = df['yield']

# split data into train & test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
y
# preview train & test sets
print('Train Set:', X_train.shape, y_train.shape)
print('Test Set:', X_test.shape, y_test.shape)
 
# build, train, & predict model
model = SVR()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# evaluate results
print('Random Forest Regressor')
print('MAE:', mean_absolute_error(y_pred, y_test))
print('MSE:', mean_squared_error(y_pred, y_test))
print('R2 Score:', r2_score(y_pred, y_test))

import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)
ax.scatter(y_test, y_pred, color='green', s=5)

# Learn more about the options like color and s and many more
ax.set_xlabel('Actual Yield')
ax.set_ylabel('Predicted Yield')
#ax.set_title('Random Forest Regression')

# Predict and calculate the scores
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
plt.grid(linewidth = 0.2)
corr = df.corr()

import seaborn as sns
sns.heatmap(corr)

import numpy as np

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
rf  =RandomForestRegressor()
param_grid = { 
    'n_estimators': [200, 500],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth' : [4,5,6,7,8],
    'criterion' :['gini', 'entropy']
}

from scipy.stats import uniform
rfc  = rfc=RandomForestRegressor(random_state=42)
CV_rfc = GridSearchCV(estimator=rfc, param_grid=param_grid, cv= 5)

# Instantiate RandomizedSearchCV model
# #rs_model = RandomizedSearchCV(RandomForestRegressor(n_jobs=-1, random_state=25),
#                               param_distributions=rf_grid,
#                               n_iter=3,
#                               cv=2,
#                               verbose=True)

# fit
CV_rfc.fit(X_train, y_train)
CV_rfc.best_params_
y_pred = rs_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print('MSE: ', mse)
print('RMSE: ', rmse)
print('R2: ', r2)

from sklearn.inspection import permutation_importance

fig, ax = plt.subplots(1, 1, figsize=(20, 20), dpi=70)
list_feature_importance = list(model.feature_importances_)
list_feature_importance

fig.suptitle('Random Forest Feature Importance', fontsize = 20)
ax.set_ylabel('Features',  fontsize=18)
ax.set_xlabel('Feature Importance', fontsize=18)
plt.figure(figsize=(20, 5))

ax.barh(y=X.columns, width=list_feature_importance)
plt.tight_layout()
plt.show()

perm_importance = permutation_importance(model, X_test, y_test)
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
sorted_idx = perm_importance.importances_mean.argsort()
fig.suptitle('Permutation Importance', fontsize = 20)
plt.bar(X.columns[sorted_idx], perm_importance.importances_mean[sorted_idx])
ax.set_xlabel('Feature Importance', fontsize=18)
y
"""